<p style="font-family: Cambria; text-align: center; font-size: 48px;"> Demography DATA Cleaning</p>

<h2>Overview</h2>
<p style="font-family: Cambria; font-size: 16px;"><b>
The dataset contains demographic information such as inpatient number, gender, weight, height, BMI, occupation, and age category. 
Several issues must be addressed before analysis:

<ul><li>Missing values</li>

<li>Implausible or incorrect values (e.g., height = 0.35 m, BMI = 404)</li>

<li>Duplicates</li>

<li>Data type inconsistencies</li>

<li>Outlier detection</li></ul>

In [80]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path


In [6]:
import warnings
warnings.simplefilter("ignore", UserWarning)

In [84]:
# Define the main project folder
project_folder = Path(
    r'C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure'
)

# Define the folder containing the original datasets
raw_data_folder = project_folder / "raw_data"

# Define the folder for cleaned datasets and reports
output_folder = project_folder / "cleaned_data"

# Create the output folder if it does not already exist
output_folder.mkdir(exist_ok=True)

print("Project Folder:", project_folder)
print("Raw Data Folder:", raw_data_folder)
print("Output Folder:", output_folder)


Project Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure
Raw Data Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\raw_data
Output Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\cleaned_data


In [85]:
# Read the original Labs dataset
df = pd.read_csv(raw_data_folder / "demography.csv")

# Keep an original copy
df_original = df.copy()

# Display the first five records
display(df.head())

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()
df.info()
df.describe()


,inpatient_number,gender,weight,height,bmi,occupation,agecat
0,5,NaN,NaN,NaN,46.000000,NaN,NaN
1,827040,Female,50.0,1.45,23.781213,NaN,69-79
2,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
3,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
4,866418,Male,70.0,1.70,24.221453,farmer,59-69


Rows: 2009
Columns: 7
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   inpatient_number  2009 non-null   int64  
 1   gender            2008 non-null   object 
 2   weight            2008 non-null   float64
 3   height            2008 non-null   float64
 4   bmi               2009 non-null   float64
 5   occupation        1981 non-null   object 
 6   agecat            2008 non-null   object 
dtypes: float64(3), int64(1), object(3)
memory usage: 110.0+ KB


,inpatient_number,weight,height,bmi
count,2009.000000,2008.000000,2008.000000,2009.000000
mean,797350.458437,52.483715,1.567869,21.803448
std,44804.294546,10.895935,0.096138,13.652299
min,5.000000,0.000000,0.350000,0.000000
25%,763040.000000,45.000000,1.500000,18.491124
50%,798725.000000,50.000000,1.560000,20.761246
75%,829366.000000,60.000000,1.620000,23.437500
max,905720.000000,115.000000,1.830000,404.081633


<p style="font-family: Cambria; font-size: 20px;"><b>Load the Dataset</b></p>

In [9]:
#Reading the file 
df = pd.read_csv(r'C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\demography.csv')

<h3>Initial Inspection</h3>
<p style="font-family: Cambria; font-size: 16px;">
We use info() to review data types, non-null values, and overall dataset structure.
This is important because it immediately shows some potential data-quality issues.
We identify missing values in each column to determine where data cleaning is required.
</p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason :</b> To know the number of rows and columns before cleaning.</p>

In [86]:
df.info()
df.describe()
df.isnull().sum()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   inpatient_number  2009 non-null   int64  
 1   gender            2008 non-null   object 
 2   weight            2008 non-null   float64
 3   height            2008 non-null   float64
 4   bmi               2009 non-null   float64
 5   occupation        1981 non-null   object 
 6   agecat            2008 non-null   object 
dtypes: float64(3), int64(1), object(3)
memory usage: 110.0+ KB


(2009, 7)

<h5><b>Actual Insight :</b> Dataset contains 2,009 records and 7 columns.</h5>

<h3>Check Data Types</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason: </b>To make sure numerical and categorical columns have the correct data type.</p>


In [87]:
df.dtypes

inpatient_number      int64
gender               object
weight              float64
height              float64
bmi                 float64
occupation           object
agecat               object
dtype: object

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight: </b>weight, height, and bmi should be numeric, while gender, occupation, and agecat are categorical.</p>

<h3>1. Remove Rows With No Useful Data</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Rows where weight, height, and BMI are all missing or zero should be removed.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason :</b>To remove records where there is no useful body-measurement information while keeping records that contain at least one of weight, height, or BMI.</p>


In [88]:
df = df[~((df['weight'].isna()) & (df['height'].isna()) & (df['bmi'].isna()))]

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight :</b>Completely blank rows do not represent meaningful patient information and can unnecessarily increase the dataset size. Removing only rows where every value is missing preserves partially completed patient records that may still contain useful demographic information.</p>

<h3>2. Check Missing Values</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We identify missing values in each column before cleaning the dataset.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason :</b> Missing values can affect analysis and calculations.</p>

In [89]:
df = df[~((df['weight'].isna()) & (df['height'].isna()) & (df['bmi'].isna()))]

df.isnull().sum()

inpatient_number     0
gender               1
weight               1
height               1
bmi                  0
occupation          28
agecat               1
dtype: int64

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Missing values exist in several demographic fields, especially occupation with 28 missing records.</p>

<h3>3. Remove Incomplete Records</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We identify duplicate rows that could result in repeated information.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>To remove rows containing missing values so that the analysis is performed only on complete records.</p>

In [90]:
df = df.dropna(
    subset=['gender', 'weight', 'height', 'agecat'],
    how='all'
)

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>The dataset contains missing values in fields such as gender, weight, height, age category, and occupation. Removing incomplete records ensures that calculations and visualizations are based on complete demographic information.</p>

<h3>4. Check Duplicate Records</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We identify duplicate rows that could result in repeated information.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Duplicate rows can incorrectly increase patient counts and distort statistics.</p>

In [91]:
df.duplicated().sum()

np.int64(0)

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Determines whether any patient information has been repeated.</p>

<h3>5. Remove Duplicate Records</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We remove completely duplicated rows from the dataset.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>To remove repeated rows so the same record is not counted more than once during analysis.</p>

In [92]:
df = df.drop_duplicates()

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Duplicate records can artificially increase patient counts and affect calculations such as averages, percentages, and frequency distributions. Removing them helps ensure that each identical record is counted only once.</p>

<h3>6. Check Duplicate Patient IDs</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We verify whether the same patient ID appears more than once.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>To check whether the same patient ID appears more than once, which could indicate repeated patient records.</p>

In [93]:
df['inpatient_number'].duplicated().sum()

np.int64(0)

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>A duplicate Patient ID does not always mean a duplicate row. The same patient may legitimately have multiple records. Therefore, first identify duplicate IDs and then check whether the associated demographic information is also repeated before removing anything.</p>

<h3>7. Standardize Text Values</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We remove unnecessary spaces from categorical columns to make values consistent.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Extra spaces can create false categories such as "Male" and " Male".</p>

In [95]:
df['gender'] = df['gender'].str.strip()
df['occupation'] = df['occupation'].str.strip()
df['agecat'] = df['agecat'].str.strip()

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Ensures demographic categories are consistently grouped.</p>

<h3>8. Handle Missing Categorical Values</h3>

<p style="font-family: Cambria; font-size: 16px;"><b>We replace missing occupation values with Unknown instead of deleting those patients.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>We should not invent a person's occupation when it is unavailable.</p>

In [96]:
df['occupation'] = df['occupation'].fillna('Unknown')

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>28 occupation values are missing, so labeling them Unknown preserves those records without creating false information.</p>

<h3>9. Check Invalid Numerical Values</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We identify zero or negative values that may represent incorrect measurements.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Weight, height and BMI should have realistic positive values.</p>

In [97]:
df[['weight', 'height', 'bmi']].le(0).sum()

weight    3
height    0
bmi       3
dtype: int64

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Helps identify impossible or suspicious measurements before analysis.</p>

<h3>10. Check Data Consistency</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We calculate BMI from weight and height to check whether the recorded BMI is consistent.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>BMI should mathematically correspond to weight and height.</p>

In [98]:
df['bmi_calculated'] = df['weight'] / (df['height'] ** 2)

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>This can reveal inconsistent BMI records. One particularly unusual BMI value is around 404, which should be investigated rather than blindly removed.</p>

<h3>11. Detect Outliers</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We use the IQR method to identify unusually high or low numerical values.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Extreme values can strongly affect averages and visualizations.</p>

In [99]:
Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)

IQR = Q3 - Q1

outliers = df[
    (df['bmi'] < Q1 - 1.5 * IQR) |
    (df['bmi'] > Q3 + 1.5 * IQR)
]

outliers

,inpatient_number,gender,weight,height,bmi,occupation,agecat,bmi_calculated
6,810128,Female,76.0,1.55,31.633715,UrbanResident,69-79,31.633715
100,842939,Male,90.0,1.64,33.462225,UrbanResident,79-89,33.462225
140,790695,Male,80.0,1.60,31.250000,UrbanResident,39-49,31.250000
169,759931,Female,75.0,1.50,33.333333,UrbanResident,79-89,33.333333
300,723617,Male,70.0,1.50,31.111111,UrbanResident,79-89,31.111111
318,837041,Female,49.0,0.48,212.673611,UrbanResident,69-79,212.673611
321,815731,Male,49.5,0.35,404.081633,farmer,49-59,404.081633
336,805044,Male,50.0,0.60,138.888889,UrbanResident,59-69,138.888889
338,789355,Female,77.5,1.58,31.044704,UrbanResident,89-110,31.044704
375,742244,Male,57.0,1.35,31.275720,UrbanResident,69-79,31.275720


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Outliers should be reviewed because they may represent either genuine extreme measurements or data-entry errors.</p>

<h3>12. Handle Missing Numerical Values</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We fill missing numerical measurements with their median when appropriate.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Median is less affected by extreme values than mean.</p>

In [100]:
df['weight'] = df['weight'].fillna(df['weight'].median())
df['height'] = df['height'].fillna(df['height'].median())

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Only the small number of missing measurements should be imputed; we avoid unnecessarily changing valid data.</p>

<h3>13. Remove Temporary Columns</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We remove the calculated BMI column after using it for data validation.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason :</b>To remove temporary/helper columns created during data cleaning that are no longer needed for the final analysis.</p>

In [101]:
df = df.drop(columns=['bmi_calculated'])
df


,inpatient_number,gender,weight,height,bmi,occupation,agecat
1,827040,Female,50.0,1.45,23.781213,Unknown,69-79
2,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
3,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
4,866418,Male,70.0,1.70,24.221453,farmer,59-69
5,775928,Male,65.0,1.70,22.491349,UrbanResident,69-79
...,...,...,...,...,...,...,...
2004,740689,Female,35.0,1.50,15.555556,Others,79-89
2005,734280,Female,50.0,1.55,20.811655,UrbanResident,79-89
2006,781004,Male,75.0,1.70,25.951557,UrbanResident,39-49
2007,744870,Male,40.0,1.50,17.777778,UrbanResident,49-59


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight :</b>If we created BMI_Calculated only to validate the existing BMI column, it should be removed after the comparison. Keeping unnecessary temporary columns can make the dataset confusing and harder to manage.</p>

<h3>14. Final Data Validation</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We verify that missing values, duplicates, and data issues have been addressed.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight :</b>To confirm the cleaning process worked.</p>

In [102]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nDuplicate Patient IDs:")
print(df['inpatient_number'].duplicated().sum())

Missing Values:
inpatient_number    0
gender              0
weight              0
height              0
bmi                 0
occupation          0
agecat              0
dtype: int64

Duplicate Rows:
0

Duplicate Patient IDs:
0


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Confirms remaining missing values are intentional, such as Unknown, and that the dataset is ready for analysis.</p>

<h3>15. Save Cleaned Dataset</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We save the cleaned dataset as a new CSV file for further analysis.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>To save the cleaned dataset as a new CSV file so the original dataset remains unchanged and the cleaned version can be used for further analysis.</p>

In [76]:
df.to_csv('demography_cleaned.csv', index=False)

In [107]:
# Save the cleaned dataset and supporting reports

# Create the output folder if it does not exist
output_folder.mkdir(exist_ok=True)

# Save cleaned dataset
df.to_csv(
    output_folder / "Team7_CodeAvengers_Category1_DataCleaning_demography.csv",
    index=False
)

# Create missing-value report
missing_report = df.isnull().sum().reset_index()
missing_report.columns = ['Column', 'Missing_Count']

# Save missing-value report
missing_report.to_csv(
    output_folder / "Team7_CodeAvengers_Category1_Missing_Value_Report_demography.csv",
    index=False
)

# Create flagged records
flagged_records = df[df.isnull().any(axis=1)].copy()

# Save flagged records
flagged_records.to_csv(
    output_folder / "Team7_CodeAvengers_Category1_Flagged_Records_demography.csv",
    index=False
)

# Display the saved files and their locations
print("All files saved successfully!\n")

for file in output_folder.iterdir():
    if file.is_file():
        print("Saved:", file.name)

print("\nOutput Folder:", output_folder)
print("Final Cleaned Dataset Shape:", df.shape)


All files saved successfully!

Saved: demography_cleaned_simple.csv
Saved: Team7_CodeAvengers_Category1_DataCleaning_demography
Saved: Team7_CodeAvengers_Category1_DataCleaning_demography.csv
Saved: Team7_CodeAvengers_Category1_Flagged_Records_demography.csv
Saved: Team7_CodeAvengers_Category1_Missing_Value_Report_demography.csv

Output Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\cleaned_data
Final Cleaned Dataset Shape: (2008, 7)


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>The cleaned dataset now contains the records retained after handling incomplete records, duplicates, invalid/incomplete measurements, and unnecessary temporary columns. Saving it separately preserves the original raw dataset for reference while providing an analysis-ready dataset.</p>

In [109]:
# Define the notebooks folder
notebook_folder = project_folder / "notebooks"

# Create the folder if it does not exist
notebook_folder.mkdir(exist_ok=True)

# Display the notebook folder location
print("Notebook Folder:", notebook_folder)

# Display existing notebooks
print("\nJupyter Notebooks Found:")

notebooks = list(notebook_folder.glob("*.ipynb"))

if notebooks:
    for notebook in notebooks:
        print(notebook.name)
else:
    print("No Jupyter Notebook found in this folder.")

Notebook Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\notebooks

Jupyter Notebooks Found:
Team7_CodeAvengers_Category1_DataCleaning_demography.ipynb


<p style="font-family: Cambria; font-size: 16px;"><b>Final Insight : </b>After data cleaning, the demography dataset contains more reliable and analysis-ready records. Duplicate records and records with no useful body-measurement information were addressed, while potentially meaningful partial records were retained. BMI outliers were detected and flagged for investigation rather than automatically removed.</p>